# Brainstorm how to Build Numerical Generator

## Final Goal: Efficiently Generate Numerical Combinations while Maintaining Problem Integrity

- [ ] For each problem, I can efficiently generate unique numerical combinations that makes sense in the problem context. All I need is to specify: Problem Index, Number of Unique Generation Required, Output Location (which stores previous generations). Then, what will happen is automated generation!


## Intermediate Goal: Justify the Numerical Digit Combinations

- [ ] To this end, I need to ensure that it is logical reasoning that we are evaluating... not arithmetic reasoning. 
  - To justify this.... I need to consider the following
      1. Intermediate numbers (1,2,3,4,5 digits)
      2. Final numbers (1,2,3,4,5 digits)
      3. Prove models are robust on these calculations


## Json Field Object as Input

{
    "problem_id": "0000",
    "generation_order": ["num_digit_1", "num_digit_2"],
    "variables": {
        "num_digit_1": {"type": "int", "range": [14, 100]},
        "num_digit_2": {"type": "int", "range": [1, 10]}
    },
    "equation_steps": {
        "num_digit_1": ["eggs_used = 3 + 4", "remainder = num_digit_1 - eggs_used"],
        "num_digit_2": ["ans = remainder * num_digit_2"]
    },
    "complexity_restrictions": {
        "remainder": {"max_digits": 2},
        "ans": {"min_digits": 1, "max_digits": 3}
    },
    "restrictions": {
        "num_digit_1": ["remainder > 0"],
        "num_digit_2": ["ans % 1 == 0"]
    }
}


## Input / Ouput / Intermediate Steps

This should be a purely arithemtic and python script. Not calling LLMs.

A class for generating numerical combinations should be constructed.

Then a script should use this class to actually build the digits combinations for specific instances. 

Input / Output would be the following:
1.  json path to the root folder for which contains all the id.json templates. (/home/mila/x/xut/github/MGSM-PRO/dataste_construction_tools/numerical_combination/template) each template correspond to a specific json problem 
I.E. under this numerical_combination I have 0000.json , 0001.json etc...

0000.json looks like this
{
    "problem_id": "0000",
    "generation_order": ["num_digit_1", "num_digit_2"],
    "variables": {
        "num_digit_1": {"type": "int", "range": [14, 100]},
        "num_digit_2": {"type": "int", "range": [1, 10]}
    },
    "equation_steps": {
        "num_digit_1": ["eggs_used = 3 + 4", "remainder = num_digit_1 - eggs_used"],
        "num_digit_2": ["ans = remainder * num_digit_2"]
    },
    "complexity_restrictions": {
        "remainder": {"max_digits": 2},
        "ans": {"min_digits": 1, "max_digits": 3}
    },
    "restrictions": {
        "num_digit_1": ["remainder > 0"],
        "num_digit_2": ["ans % 1 == 0"]
    }
}

2. the problem ID I want to construct numerical combinations for (so like for th 0000.json case, it will be 0. If I want multiple, it will be [0, 15] which will mean like all from 0000.json to 0015.json)

3. number of numerical combinations I want to construct for each. i.e. I say 10, then 10 unique ones need to be created. Unique numerical combiantion means like the variables (num_digit_...) cannot all be the same, though they might be similar. 

4. output folder (/home/mila/x/xut/github/MGSM-PRO/dataste_construction_tools/numerical_combination/combination). This is where I store all the numerical combinations. They will also be organized as files like 0000.jsonl.
Each line is a new json object with like 
{combination_instance: 1, variable_values: {"num_digit_1" : zzz , "num_digit_2": bbb, "ans": aaa}}
Note: everytime we are trying to generate new numerical combiations, we first load this, and make sure none of the new generated instances are the same as the ones we generated before. 

5. number of trials (i.e. 10000). This is the number of trials that cpu will take before giving up.

Intermediate steps:
If we look at the 0000.json again.
{
    "problem_id": "0000",
    "generation_order": ["num_digit_1", "num_digit_2"],
    "variables": {
        "num_digit_1": {"type": "int", "range": [14, 100]},
        "num_digit_2": {"type": "int", "range": [1, 10]}
    },
    "equation_steps": {
        "num_digit_1": ["eggs_used = 3 + 4", "remainder = num_digit_1 - eggs_used"],
        "num_digit_2": ["ans = remainder * num_digit_2"]
    },
    "complexity_restrictions": {
        "remainder": {"max_digits": 2},
        "ans": {"min_digits": 1, "max_digits": 3}
    },
    "restrictions": {
        "num_digit_1": ["remainder > 0"],
        "num_digit_2": ["ans % 1 == 0"]
    }
}

It specifies a lot of things.

1. Generation order. You must generate each step by step. following the order of the list. In this case, generate num_digit_1 first, then num_digit_2.

2. To generate the varaibles, look at the "variables" and find its corresponding one there. I.E. the "num_digit_1". Corresponding to it will be a dictionary {}. The type is the sort of values you need to generate. The range is where you pick the variables from. If it is not range but "construction", this means that you don't need to randomly pick this value. Rather, you will be constructing this value from pre-existing values. I.E. if "num_digit_2": {"type": "int", "construction": "num_digit_2 = 2 * num_digit_1"}, you just take num_digit_1 and construct num_digit_2

3. After generating a num_digit, the first thing you do is go to equation steps. There will be a list corresponding to each varaible. Operate the list step by step, from the first to the last. You should keep track of EVERY variable that there is in the calculation that you have in your combination (ALSO keep track of those that are pre-existing in the problem calculation that might not correspond to variables (i.e. the intermediate variables such as eggs_used)) 

4. After the equation steps of the generated corresponding digit, you go to the restrictions. There is again very similar, a list and you check one by one with the current variables you have. Note, all of string operations should already have the variables from which you need genereated. So you just check. If a check fails. just restart the generation process

5. After this, head to complexity restrictions AND make sure the intermediate variables follow the complexity restrictions. If not, restart the genreation process again.

6. Proceed to the generation of the second variable now... and repeat the process until generation is done. 

7. If everything passes, you should append this in the output with the ans alongside the variables that were picked. 





